In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
print("Key found:", os.getenv("ALPACA_KEY") is not None)
print("Secret found:", os.getenv("ALPACA_SECRET") is not None)

Key found: True
Secret found: True


In [2]:
from alpaca.data.historical import StockHistoricalDataClient

client = StockHistoricalDataClient(os.getenv("ALPACA_KEY"), os.getenv("ALPACA_SECRET"))
print("Client ready")

Client ready


In [3]:
key = os.getenv("ALPACA_KEY")
secret = os.getenv("ALPACA_SECRET")
print("Key starts with:", key[:2], "| key length:", len(key), "| secret length:", len(secret))
print("Has spaces or quotes:", any(c in key + secret for c in " \"'"))

Key starts with: PK | key length: 26 | secret length: 44
Has spaces or quotes: False


In [4]:
from datetime import datetime
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.data.enums import DataFeed

request = StockBarsRequest(
    symbol_or_symbols=["AAPL", "MSFT"],
    timeframe=TimeFrame(5, TimeFrameUnit.Minute),
    start=datetime(2026, 9, 17),
    end=datetime(2026, 9, 19),
    feed=DataFeed.IEX,
)

In [5]:
bars = client.get_stock_bars(request)
df = bars.df.reset_index()

print(len(df), "rows")
df.head()

336 rows


,symbol,timestamp,open,high,low,close,volume,trade_count,vwap
0,AAPL,2026-09-17 12:30:00+00:00,334.53,334.55,334.50,334.50,295.0,8.0,334.513913
1,AAPL,2026-09-17 12:35:00+00:00,334.30,334.30,334.30,334.30,90.0,2.0,334.300000
2,AAPL,2026-09-17 12:40:00+00:00,334.33,334.33,334.33,334.33,60.0,2.0,334.330000
3,AAPL,2026-09-17 12:45:00+00:00,334.20,334.20,334.20,334.20,60.0,2.0,334.200000
4,AAPL,2026-09-17 12:50:00+00:00,334.10,334.11,334.10,334.11,333.0,6.0,334.107046


In [8]:
from datetime import time

df["ny_time"] = df["timestamp"].dt.tz_convert("America/New_York")
regular = df[(df["ny_time"].dt.time >= time(9, 30)) & (df["ny_time"].dt.time < time(16, 0))]

regular.groupby(["symbol", regular["ny_time"].dt.date]).size().reset_index(name="bars")

,symbol,ny_time,bars
0,AAPL,2026-09-17,78
1,AAPL,2026-09-18,78
2,MSFT,2026-09-17,78
3,MSFT,2026-09-18,78


In [7]:
display(df.head())
display(regular.head())

,symbol,timestamp,open,high,low,close,volume,trade_count,vwap,ny_time
0,AAPL,2026-09-17 12:30:00+00:00,334.53,334.55,334.50,334.50,295.0,8.0,334.513913,2026-09-17 08:30:00-04:00
1,AAPL,2026-09-17 12:35:00+00:00,334.30,334.30,334.30,334.30,90.0,2.0,334.300000,2026-09-17 08:35:00-04:00
2,AAPL,2026-09-17 12:40:00+00:00,334.33,334.33,334.33,334.33,60.0,2.0,334.330000,2026-09-17 08:40:00-04:00
3,AAPL,2026-09-17 12:45:00+00:00,334.20,334.20,334.20,334.20,60.0,2.0,334.200000,2026-09-17 08:45:00-04:00
4,AAPL,2026-09-17 12:50:00+00:00,334.10,334.11,334.10,334.11,333.0,6.0,334.107046,2026-09-17 08:50:00-04:00


,symbol,timestamp,open,high,low,close,volume,trade_count,vwap,ny_time
10,AAPL,2026-09-17 13:30:00+00:00,334.805,335.390,332.620,332.665,26767.0,590.0,333.770522,2026-09-17 09:30:00-04:00
11,AAPL,2026-09-17 13:35:00+00:00,332.550,332.880,331.400,332.850,24610.0,596.0,332.042564,2026-09-17 09:35:00-04:00
12,AAPL,2026-09-17 13:40:00+00:00,332.910,332.990,331.850,331.935,17633.0,333.0,332.248471,2026-09-17 09:40:00-04:00
13,AAPL,2026-09-17 13:45:00+00:00,332.050,332.210,330.775,330.910,15714.0,455.0,331.441573,2026-09-17 09:45:00-04:00
14,AAPL,2026-09-17 13:50:00+00:00,331.045,331.045,330.230,330.630,37871.0,825.0,330.500315,2026-09-17 09:50:00-04:00


In [9]:
df.drop(columns="ny_time").to_csv("data/raw/pilot_AAPL_MSFT_2026-09-17_18_iex.csv", index=False)

In [11]:
import pandas_market_calendars as mcal

nyse = mcal.get_calendar("NYSE")
schedule = nyse.schedule(start_date="2026-07-15", end_date="2026-09-18")
schedule.tail()

,market_open,market_close
2026-09-14,2026-09-14 13:30:00+00:00,2026-09-14 20:00:00+00:00
2026-09-15,2026-09-15 13:30:00+00:00,2026-09-15 20:00:00+00:00
2026-09-16,2026-09-16 13:30:00+00:00,2026-09-16 20:00:00+00:00
2026-09-17,2026-09-17 13:30:00+00:00,2026-09-17 20:00:00+00:00
2026-09-18,2026-09-18 13:30:00+00:00,2026-09-18 20:00:00+00:00


In [12]:
last25 = schedule.tail(25)
calibration_days = last25.head(5)
main_days = last25.tail(20)

print("Calibration:", [d.strftime("%b %d") for d in calibration_days.index])
print("Main:", main_days.index[0].strftime("%b %d"), "to", main_days.index[-1].strftime("%b %d"))
print("Early closes:", len(nyse.early_closes(schedule)))

Calibration: ['Aug 14', 'Aug 17', 'Aug 18', 'Aug 19', 'Aug 20']
Main: Aug 21 to Sep 18
Early closes: 0


In [13]:
import pandas as pd

candidates = pd.read_csv("data/candidate_stocks.csv")
tickers = candidates["ticker"].tolist()
print(len(tickers), "tickers:", tickers[:5])

60 tickers: ['AAPL', 'MSFT', 'NVDA', 'ORCL', 'CSCO']


In [20]:
def download_one(ticker):
    request = StockBarsRequest(
        symbol_or_symbols=ticker,
        timeframe=TimeFrame(5, TimeFrameUnit.Minute),
        start=datetime(2026, 8, 14),
        end=datetime(2026, 9, 19),
        feed=DataFeed.SIP,
    )
    df = client.get_stock_bars(request).df.reset_index()
    df.to_csv(f"data/raw/bars_{ticker}.csv", index=False)
    return len(df)

In [15]:
download_one("AAPL")

2058

In [21]:
import os

failed = []
for ticker in tickers:
    if os.path.exists(f"data/raw/bars_{ticker}.csv"):
        continue
    try:
        print(ticker, download_one(ticker), "rows")
    except Exception as e:
        print(ticker, "FAILED:", e)
        failed.append(ticker)

print("Done. Failed:", failed)

AAPL 4791 rows
MSFT 4693 rows
NVDA 4800 rows
ORCL 4710 rows
CSCO 3436 rows
AKAM 2306 rows
GOOGL 4779 rows
META 4756 rows
VZ 3001 rows
DIS 2697 rows
NYT 2042 rows
AMZN 4700 rows
HD 2947 rows
MCD 3794 rows
NKE 4609 rows
SBUX 2591 rows
HAS 2022 rows
PG 2610 rows
KO 3522 rows
PEP 3102 rows
COST 2879 rows
CLX 2375 rows
JNJ 2545 rows
UNH 3323 rows
LLY 3132 rows
MRK 2712 rows
GILD 2245 rows
INCY 2072 rows
JPM 3228 rows
BAC 3443 rows
V 2855 rows
GS 2700 rows
PYPL 3447 rows
TROW 2029 rows
CAT 2990 rows
GE 2814 rows
UPS 2639 rows
DE 2263 rows
PCAR 2053 rows
JBHT 2031 rows
XOM 3457 rows
CVX 3319 rows
COP 2588 rows
SLB 2580 rows
APA 2531 rows
NEE 2496 rows
DUK 2204 rows
SO 2139 rows
AEP 2169 rows
PNW 1985 rows
PLD 2121 rows
AMT 2065 rows
O 3196 rows
EQIX 1996 rows
REG 2011 rows
LIN 2245 rows
NEM 3348 rows
FCX 3568 rows
SHW 2128 rows
MOS 2780 rows
Done. Failed: []


In [22]:
import glob

files = glob.glob("data/raw/bars_*.csv")
raw = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

raw["ny_time"] = pd.to_datetime(raw["timestamp"], utc=True).dt.tz_convert("America/New_York")
reg = raw[(raw["ny_time"].dt.time >= time(9, 30)) & (raw["ny_time"].dt.time < time(16, 0))]
counts = reg.groupby(["symbol", reg["ny_time"].dt.date]).size().reset_index(name="bars")
print(len(files), "files,", len(counts), "stock-days")

60 files, 1500 stock-days


In [23]:
summary = counts.groupby("symbol").agg(
    days=("bars", "size"),
    min_bars=("bars", "min"),
    good_days=("bars", lambda b: (b >= 75).sum()),
)
summary.sort_values("good_days").head(15)

,days,min_bars,good_days
symbol,,,
JBHT,25,70,21
AAPL,25,78,25
META,25,78,25
MOS,25,78,25
MRK,25,78,25
MSFT,25,78,25
NEE,25,78,25
NEM,25,78,25
NKE,25,78,25


In [19]:
test = StockBarsRequest(symbol_or_symbols="REG", timeframe=TimeFrame(5, TimeFrameUnit.Minute),
                        start=datetime(2026, 8, 14), end=datetime(2026, 9, 19), feed=DataFeed.SIP)
sip = client.get_stock_bars(test).df.reset_index()

sip["ny_time"] = sip["timestamp"].dt.tz_convert("America/New_York")
sip_reg = sip[(sip["ny_time"].dt.time >= time(9, 30)) & (sip["ny_time"].dt.time < time(16, 0))]
sip_reg.groupby(sip_reg["ny_time"].dt.date).size().describe()

count    25.000000
mean     77.880000
std       0.331662
min      77.000000
25%      78.000000
50%      78.000000
75%      78.000000
max      78.000000
dtype: float64

In [24]:
import json

final = candidates[candidates["ticker"] != "JBHT"]

config = {
    "tickers": final["ticker"].tolist(),
    "excluded": {"JBHT": "failed the 95% coverage rule on 4 of 25 days (SIP feed)"},
    "feed": "sip",
    "timeframe": "5Min",
    "calibration_days": ["2026-08-14", "2026-08-17", "2026-08-18", "2026-08-19", "2026-08-20"],
    "main_start": "2026-08-21",
    "main_end": "2026-09-18",
    "regular_hours_ny": ["09:30", "16:00"],
    "coverage_threshold": 0.95,
}

with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

print(len(config["tickers"]), "tickers saved")

59 tickers saved
